# **Project 1: Extract the Employment Act PDF**

**Goal:** turn the PDF into a clean list of sections, each with a number, a title and its text.

Steps in this notebook:
1. Read the pages
2. Clean one page (remove header, footer, footnotes)
3. Join all pages into one list of lines
4. Read the table of contents
5. Split the body into sections
6. Remove the next section's title from the end of each section
7. Turn each section's lines into text
8. Check the result
9. Save it for the next notebook

Rule for this notebook: run a cell, **look at the output**, and only then go on.

In [15]:
from pypdf import PdfReader

PDF_PATH = "../data/employment-act-1955.pdf"

pages = [p.extract_text() or "" for p in PdfReader(PDF_PATH).pages]

print(len(pages))
print(pages[22])

121
Employment
23
9. (Deleted by *Act 40 of 1966 ).
Contracts to be in writing and to include provision for 
termination
10. (1) A contract of service for a specified period of time 
exceeding one month or for the performance of a specified  
piece of work, where the time reasonably required for the 
completion of the work exceeds or may exceed one month, shall 
be in writing.
 (2) In every written contract of service a clause shall be 
included setting out the manner in which such contract may be 
terminated by either party in accordance with this Part.
Provision as to termination of contracts
11. (1) A contract of service for a specified period of time or 
for the performance of a specified piece of work shall, unless 
otherwise terminated in accordance with this Part, terminate when 
the period of time for which such contract was made has expired 
or when the piece of work specified in such contract has been 
completed.
 (2) A contract of service for an unspecified period of time 
s

### **Step 1: Look at the raw page**
Write down what you see: what repeats on every page (header, footer), where the section title is, and how footnotes start.

### **Step 2: Clean one page**
`pypdf` can split the page header over several lines, so we remove up to 4 header lines from the top of each page.

In [16]:
def is_header_line(line):
    parts = line.split()
    return (line.isdigit()                                   # a lone page number
            or line in ("Employment", "Laws of Malaysia", "Act 265")   # header split over lines
            or (len(parts) == 2 and parts[0] == "Employment" and parts[1].isdigit())
            or line.endswith("Laws of Malaysia Act 265"))    # even-page header on one line

def clean_page(text):
    raw = [l.strip() for l in text.splitlines() if l.strip()]

    # page header: at most the first 4 lines
    for _ in range(4):
        if raw and is_header_line(raw[0]):
            raw.pop(0)

    kept = []
    for line in raw:
        if line.startswith("*NOTE"):          # footnotes: drop the rest of the page
            break
        if line.startswith("WJW"):            # footer line
            continue
        if line[0].isdigit() and ("(Deleted" in line or "(Omitted" in line):
            continue                          # lines like "9. (Deleted by Act ...)"
        kept.append(line)
    return kept

clean_page(pages[22])

['Contracts to be in writing and to include provision for',
 'termination',
 '10. (1) A contract of service for a specified period of time',
 'exceeding one month or for the performance of a specified',
 'piece of work, where the time reasonably required for the',
 'completion of the work exceeds or may exceed one month, shall',
 'be in writing.',
 '(2) In every written contract of service a clause shall be',
 'included setting out the manner in which such contract may be',
 'terminated by either party in accordance with this Part.',
 'Provision as to termination of contracts',
 '11. (1) A contract of service for a specified period of time or',
 'for the performance of a specified piece of work shall, unless',
 'otherwise terminated in accordance with this Part, terminate when',
 'the period of time for which such contract was made has expired',
 'or when the piece of work specified in such contract has been',
 'completed.',
 '(2) A contract of service for an unspecified period of time

### **Debug view (optional, use it while you fix the rules)**
`show` prints numbered lines. `show_tagged` labels every line as KEEP / HEADER / FOOTER / NOTE / DELETED so you can see *what was removed*.

In [17]:
def show(lines, start=0, n=30):
    for i, line in enumerate(lines[start:start + n], start):
        print(f"{i:5} | {line}")

def tag_page(text):
    out = []
    raw = [l.strip() for l in text.splitlines() if l.strip()]
    in_header = True
    in_note = False
    for line in raw:
        if in_header and is_header_line(line):
            out.append(("HEADER", line))
            continue
        in_header = False
        if line.startswith("WJW"):
            out.append(("FOOTER", line))
        elif line.startswith("*NOTE"):
            in_note = True
            out.append(("NOTE", line))
        elif in_note:
            out.append(("NOTE", line))
        elif line[0].isdigit() and ("(Deleted" in line or "(Omitted" in line):
            out.append(("DELETED", line))
        else:
            out.append(("KEEP", line))
    return out

def show_tagged(page_no):
    for tag, line in tag_page(pages[page_no]):
        print(f"{tag:8}| {line}")

show_tagged(22)

HEADER  | Employment
HEADER  | 23
DELETED | 9. (Deleted by *Act 40 of 1966 ).
KEEP    | Contracts to be in writing and to include provision for
KEEP    | termination
KEEP    | 10. (1) A contract of service for a specified period of time
KEEP    | exceeding one month or for the performance of a specified
KEEP    | piece of work, where the time reasonably required for the
KEEP    | completion of the work exceeds or may exceed one month, shall
KEEP    | be in writing.
KEEP    | (2) In every written contract of service a clause shall be
KEEP    | included setting out the manner in which such contract may be
KEEP    | terminated by either party in accordance with this Part.
KEEP    | Provision as to termination of contracts
KEEP    | 11. (1) A contract of service for a specified period of time or
KEEP    | for the performance of a specified piece of work shall, unless
KEEP    | otherwise terminated in accordance with this Part, terminate when
KEEP    | the period of time for which such cont

In [18]:
# does the debug view agree with the real clean_page on every page?
bad = [i for i, p in enumerate(pages)
       if [l for t, l in tag_page(p) if t == "KEEP"] != clean_page(p)]
print("pages where the two versions disagree:", bad)

# which pages have footnotes? look at one of them
note_pages = [i for i, p in enumerate(pages) if "*NOTE" in p]
print(note_pages[:5])

pages where the two versions disagree: []
[10, 22, 40, 41, 46]


### **Step 3: Join all pages into one list of lines**
A section can start on one page and end on the next, so from here on we ignore pages.

In [19]:
lines = []

for page in pages:
    lines.extend(clean_page(page))

print(len(lines))
show(lines, 0, 30)

4012
    0 | LAWS OF MALAYSIA
    1 | REPRINT
    2 | Act 265
    3 | EMPLOYMENT ACT 1955
    4 | As at 1 August 2023
    5 | PUBLISHED  BY
    6 | THE COMMISSIONER OF LAW REVISION, MALAYSIA
    7 | UNDER THE AUTHORITY OF THE REVISION OF LAWS ACT 1968
    8 | 2023
    9 | EMPLOYMENT ACT 1955
   10 | First enacted … … … … 1955 (F.M. Ordinance
   11 | No. 38 of 1955)
   12 | Revised … … … … 1981 (Act 265 w.e.f.
   13 | 18 February 1982)
   14 | Latest amendment made by
   15 | Act A1651 which came
   16 | into operation on   … … 1 January 2023
   17 | Previous  rePrints
   18 | First Reprint ... ... ... 1975
   19 | Second Reprint ... ... ... 2001
   20 | Third Reprint ... ... ... 2006
   21 | LAWS OF MALAYSIA
   22 | Act 265
   23 | EMPLOYMENT ACT 1955
   24 | ARRANGEMENT OF SECTIONS
   25 | Part I
   26 | PRELIMINARY
   27 | Section
   28 | 1. Short title and application
   29 | 2. Interpretation


### **Step 4: Read the table of contents**
The table of contents lists every section number and title, so we use it as the list of valid sections.
`partition(". ")` cuts a line at the first ". " into 3 pieces: before, the separator, after.

In [20]:
def toc_entry(line):
    # a table-of-contents line -> (number, title), or None if it is not an entry
    number, dot, title = line.partition(". ")
    if dot and number[:1].isdigit() and number.isalnum() and len(number) <= 5:
        return number, title
    return None

print(toc_entry("60a. Hours of work and working at night"))
print(toc_entry("Part II"))

('60a', 'Hours of work and working at night')
None


In [21]:
import re

# --- Cell 6 & 7 Replacement: Parse TOC cleanly ---
start = lines.index("ARRANGEMENT OF SECTIONS")
end = next(i for i, l in enumerate(lines) if l.startswith("An Act relating to employment"))
toc_lines = lines[start + 1:end]

toc = {}      # "60a" -> "Hours of work and working at night"
order = []    # ["1", "2", "2a", ...] in document order
last = None

for line in toc_lines:
    skip = (line == "Section"
            or line == "Act 265"
            or line in ("First Schedule", "Second Schedule")
            or (line.startswith("Part ") and len(line) < 12)
            or (line.isupper() and len(line) > 3))
    if skip:
        continue

    # 1. Collapse spaces inside section numbers: "60 b." -> "60b.", "81 a." -> "81a."
    cleaned = re.sub(r'\b(\d+)\s+([a-zA-Z]{1,3})\.', r'\1\2.', line)

    # 2. Split lines containing multiple TOC entries (e.g. "...night 60b. Task work 60c. Shift work")
    parts = re.split(r'(?=\b\d+[a-zA-Z]{0,3}\.\s+)', cleaned)

    for part in parts:
        part = part.strip()
        if not part:
            continue
        number, dot, title = part.partition(". ")
        if dot and number[:1].isdigit() and number.isalnum() and len(number) <= 5:
            last = number.lower()
            toc[last] = title.strip()
            order.append(last)
        elif last:
            toc[last] += " " + part

### **Step 5: Split the body into sections**
A body line starts a new section only if its number is in the table of contents **and** comes next in order.
That rule stops ordinary lines that happen to start with a number from being treated as sections.

In [22]:
def section_start(line, toc):
    # a body line -> (number, rest of line) if it starts a known section, else None
    number, dot, rest = line.partition(". ")
    if dot and number in toc:
        return number, rest
    return None

In [23]:
body = lines[end + 1:]
sections = []
current = None
pos = -1

for line in body:
    # Only stop on standalone Schedule headings at the end of the Act, not body text mentioning "First Schedule"
    if line.strip().upper() in ("FIRST SCHEDULE", "SECOND SCHEDULE"):
        break

    norm_line = re.sub(r'^(\d+)\s+([a-zA-Z]{1,3})\.', r'\1\2.', line)
    number, dot, rest = norm_line.partition(". ")

    if dot and number.lower() in toc:
        num_key = number.lower()
        idx = order.index(num_key)
        if 0 < idx - pos <= 5:
            if current:
                sections.append(current)
            current = {"no": num_key.upper(),
                       "title": toc[num_key],
                       "lines": [rest]}
            pos = idx
            continue

    if current:
        current["lines"].append(line)

if current:
    sections.append(current)

print(len(sections), len(order))

130 134


### **Step 6: Remove the next section's title from the end of each section**
In this PDF the title is printed on the line *before* its section number, so it ends up at the bottom of the previous section.

In [24]:
def norm(s):
    return " ".join(s.split()).lower()

unmatched = []

for i in range(len(sections) - 1):
    ls = sections[i]["lines"]
    next_title = norm(sections[i + 1]["title"])

    removed = False
    for k in range(1, 5):                              # the title may wrap over up to 4 lines
        if norm(" ".join(ls[-k:])) == next_title:
            del ls[-k:]
            removed = True
            break
    if not removed:
        unmatched.append(sections[i + 1]["no"])

    # also drop Part headings such as "Part II" + "CONTRACTS OF SERVICE" left at the end
    while ls and ((ls[-1].startswith("Part ") and len(ls[-1]) < 12)
                  or (ls[-1].isupper() and len(ls[-1]) > 3)):
        ls.pop()

print("titles that could not be removed:", unmatched)

titles that could not be removed: ['44A', '69A', '81', '81G', '93', '103']


### **Step 7: Turn each section's lines into text**
The PDF breaks lines in the middle of sentences. We join them back, but start a new line at each subsection like (1), (2a).

In [25]:
def join_lines(lines):
    out = []
    for line in lines:
        if not out or (line.startswith("(") and line[1:2].isdigit()):
            out.append(line)           # a subsection starts a new line
        else:
            out[-1] += " " + line      # otherwise the sentence continues
    return "\n".join(out)

for s in sections:
    s["text"] = join_lines(s["lines"])

### **Step 8: Check the result**
Three things to check: `missing` is empty or short, s.12 has the four / six / eight weeks notice rule, and s.60A starts with the working hours rule.

In [26]:
found_nos = {s["no"].lower() for s in sections}
print("missing:", [n for n in order if n not in found_nos])

by_no = {s["no"]: s for s in sections}
print("unique numbers:", len(by_no) == len(sections))

print(by_no["12"]["title"])
print(by_no["12"]["text"][:700])

missing: ['60g', '60h', '60l', '69e']
unique numbers: True
Notice of termination of contract
(1) Either party to a contract of service may at any time give to the other party notice of his intention to terminate such contract of service.
(2) The length of such notice shall be the same for both employer and employee and shall be determined by a provision made in writing for such notice in the terms of the contract of service, or, in the absence of such provision in writing, shall not be less than— (a) four weeks’ notice if the employee has been so employed for less than two years on the date on which the notice is given; (b) six weeks’ notice if he has been so employed for two years or more but less than five years on such date; (c) eight weeks’ notice if he has been so employed for 


In [27]:
print(by_no["60A"]["title"])
print(by_no["60A"]["text"][:900])

Hours of work and working at night
(1) Except as hereinafter provided, an employee shall not be required under his contract of service to work— (a) more than five consecutive hours without a period of leisure of not less than thirty minutes duration; (b) more than eight hours in one day; (c) in excess of a spread over period of ten hours in one day; (d) more than forty-five hours in one week: Provided that— (i) for the purpose of paragraph (1)(a), any break of less than thirty minutes in the five consecutive hours shall not break the continuity of that five consecutive hours; (ii) an employee who is engaged in work which must be carried on continuously and which requires his continual attendance may be required to work for eight consecutive hours inclusive of a period or periods of not less than forty-five minutes in the aggregate during which he shall have the opportunity to have a meal; and (iii) where, by agreement und


In [28]:
# overview table: very short sections may have lost text, very long ones may have swallowed the next section
import pandas as pd

df = pd.DataFrame([{"no": s["no"], "title": s["title"][:40], "chars": len(s["text"])} for s in sections])
print(df.sort_values("chars").head(5))
print(df.sort_values("chars", ascending=False).head(5))

      no                                     title  chars
47   44A                               ( Deleted )     25
117   93                               ( Deleted )     25
100  81G                               ( Deleted )     25
0      1               Short title and application    108
65   60M  Prohibition on termination of local for     120
      no                                     title  chars
129  103  Repeal and saving f irst  schedule s eco  15130
1      2                            Interpretation  12186
54   60A        Hours of work and working at night   6756
38    37  Length of eligible period and entitlemen   4762
102   82                        Service of summons   4743


### **Step 9: Save the sections for the next notebook**

In [29]:
import json

with open("../data/sections.json", "w", encoding="utf-8") as f:
    json.dump([{"no": s["no"], "title": s["title"], "text": s["text"]} for s in sections],
              f, ensure_ascii=False, indent=2)

print("saved", len(sections), "sections")

saved 130 sections
